In [ ]:
import os
import json

import dask.dataframe as dd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns

### Setup

In [ ]:
# Order of all possible bins for plotting
order = [
    # '(0, 10]', 
         '(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]', '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 500]', '(500, 1000]']

bins = [
    # 0, 
        10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 150, 200, 500, 1000]

### RiverSP gauge comparison

In [ ]:
data_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results/'
save_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results_figures/'

In [ ]:
# Read in gauge obs that we already filtered in 01_parse_wse
df_sp = pd.read_parquet(os.path.join(data_dir, 'harlan_wse_observations.parquet'))

print(df_sp.SWOT_data_version.unique())

# Extract widths for NHD comparison
df_p_width = df_sp[['gage_id', 'p_width']].drop_duplicates()
df_p_width.columns = ['site_id', 'p_width']

### PIXC gauge comparison

In [ ]:
# Read in gauge obs that we already filtered in 01_parse_wse
df = pd.read_parquet(os.path.join(data_dir, 'bennitt_wse_observations.parquet'))

# Calculate the correlation for the full dataset
grouped_corr_full = df.groupby(['Bin', 'site_id'], observed=False)[['wse_swot_norm', 'wse_gauge_norm']].corr()
corr_df_full = grouped_corr_full.unstack().iloc[:, 1].reset_index()
corr_df_full.columns = ['Bin', 'site_id', 'correlation']

In [ ]:
with open("./data/filtered_ids.json", "w", encoding="utf-8") as file:
    json.dump(list(corr_df_full.site_id.unique()), file, indent=4)

# Calculate the bias for only gauges with NAVD 88
mdata_dir = '/nas/cee-water/cjgleason/fiona/USGS_gauges/'
gauges = pd.read_parquet(path=os.path.join(mdata_dir,
                                           'usgs_stations_with_stage_and_nhd_metadata.parquet'),
                         columns=['site_id', 'alt_datum_cd'])

df = pd.merge(df, gauges, on='site_id')

df_navd_88 = df[df['alt_datum_cd'] == 'NAVD88']

diff_mean = df_navd_88.groupby(['Bin', 'site_id'], observed=False)['wse_diff'].mean().reset_index()
diff_mean = diff_mean[~diff_mean.wse_diff.isna()].reset_index().drop(columns='index')
diff_mean.columns = ['Bin', 'site_id', 'wse_diff_mean']

diff_median = df_navd_88.groupby(['Bin', 'site_id'], observed=False)['wse_diff'].median().reset_index()
diff_median = diff_median[~diff_median.wse_diff.isna()].reset_index().drop(columns='index')
diff_median.columns = ['Bin', 'site_id', 'wse_diff_median']

bias_df_full = pd.merge(diff_mean, diff_median, on=['site_id', 'Bin'])

In [ ]:
# Extract widths for NHD comparison
df_nhd_width = df[['site_id', 'WidthM']].drop_duplicates()
# df_nhd_width.columns = ['site_id', 'p_width']

# Merge on p_width (removes non-overlapping gauges)
df = df.merge(df_p_width, how='inner', on='site_id')

df['Bin'] = pd.cut(df['p_width'], bins).astype(str)
df['Bin'] = pd.Categorical(df['Bin'], categories=order, ordered=True)
df = df.reset_index()
df = df.drop(columns='index')

In [ ]:
bennitt_ids = set(df.site_id)
harlan_ids = set(df_sp.gage_id)
# harlan_ids = set(df_sp.site_id)

match_ids = bennitt_ids.intersection(harlan_ids)

In [ ]:
with open("./data/harlan_bennitt_match_ids.json", "w", encoding="utf-8") as file:
    json.dump(list(match_ids), file, indent=4)

# # Subset Merritt's gauges to just the intersection
df_sp = df_sp[df_sp['gage_id'].isin(match_ids)]

### Calculate correlation and bias

In [ ]:
# Correlation for matched gauges, this study
grouped_corr = df.groupby(['Bin', 'site_id'], observed=False)[['wse_gauge_norm', 'wse_swot_norm']].corr()
corr_df = grouped_corr.unstack().iloc[:, 1].reset_index()
corr_df.columns = ['Bin', 'site_id', 'correlation']

In [ ]:
# Bias for this study, matched gauges
df_navd_88 = df[df['alt_datum_cd'] == 'NAVD88']

diff_mean = df_navd_88.groupby(['Bin', 'site_id'], observed=False)['wse_diff'].mean().reset_index()
diff_mean = diff_mean[~diff_mean.wse_diff.isna()].reset_index().drop(columns='index')
diff_mean.columns = ['Bin', 'site_id', 'wse_diff_mean']

diff_median = df_navd_88.groupby(['Bin', 'site_id'], observed=False)['wse_diff'].median().reset_index()
diff_median = diff_median[~diff_median.wse_diff.isna()].reset_index().drop(columns='index')
diff_median.columns = ['Bin', 'site_id', 'wse_diff_median']

bias_df = pd.merge(diff_mean, diff_median, on=['site_id', 'Bin'])

In [ ]:
# Correlation for matched gauges, Harlan et al. 2026
grouped_corr_sp = df_sp.groupby(['Bin', 'gage_id'], observed=False)[['wse_gauge_norm', 'wse_swot_norm']].corr()
corr_df_sp = grouped_corr_sp.unstack().iloc[:, 1].reset_index()
corr_df_sp.columns = ['Bin', 'site_id', 'correlation']

In [ ]:
# Bias for this study, Harlan et al. 2026
df_navd_88 = df_sp[df_sp['gage_alt_datum'] == 'NAVD88']

diff_mean = df_navd_88.groupby(['Bin', 'gage_id'], observed=False)['wse_diff'].mean().reset_index()
diff_mean = diff_mean[~diff_mean.wse_diff.isna()].reset_index().drop(columns='index')
diff_mean.columns = ['Bin', 'site_id', 'wse_diff_mean']

diff_median = df_navd_88.groupby(['Bin', 'gage_id'], observed=False)['wse_diff'].median().reset_index()
diff_median = diff_median[~diff_median.wse_diff.isna()].reset_index().drop(columns='index')
diff_median.columns = ['Bin', 'site_id', 'wse_diff_median']

bias_df_sp = pd.merge(diff_mean, diff_median, on=['site_id', 'Bin'])

In [ ]:
corr_df_full['data'] = 'This study: filtered sites (NHDPlus HR widths)'
corr_df['data'] = 'This study: validation sites (SWORD widths)' 
corr_df_sp['data'] = 'Harlan et al. 2026: validation sites (SWORD widths)'

bias_df_full['data'] = 'This study: filtered sites (NHDPlus HR widths)'
bias_df['data'] = 'This study: validation sites (SWORD widths)'
bias_df_sp['data'] = 'Harlan et al. 2026: validation sites (SWORD widths)'

# Create combined correlation df
corr_combo = pd.concat(
    [corr_df_full, corr_df, corr_df_sp]
).reset_index()
corr_combo = corr_combo.drop(columns='index')

counts_df = corr_combo.groupby(['Bin', 'data'], observed=False)['correlation'].count().unstack()

In [ ]:
# Create combined correlation df
bias_combo = pd.concat(
    [bias_df_full, bias_df, bias_df_sp]
).reset_index()
bias_combo = bias_combo.drop(columns='index')

# Create the counts table
counts_bias_df = bias_combo.pivot_table(
    index='Bin', 
    columns='data', 
    values=['wse_diff_mean', 'wse_diff_median'], 
    aggfunc='count',
    observed=False
)

In [ ]:
# 1. Filter data upfront for just the requested study
target_study = 'This study: filtered sites (NHDPlus HR widths)'
study_df = corr_combo[corr_combo['data'] == target_study]

# 2. Apply your sample size filter (>= 20) and bin filtering
plot_df = study_df#.groupby('Bin', observed=False).filter(lambda x: len(x) >= 20)
# plot_df = plot_df[plot_df['Bin'].isin(order[0:-3])]

# Correlation plot
fig, ax = plt.subplots(figsize=(19, 9))
sns.set_style('whitegrid')

# Single color from your original palette choice
# plot_color = '#668D6B' 
plot_color = sns.color_palette('tab20b')[-3]

# 3. Plot single-color boxplot (removed hue parameter)
sns.boxplot(
    data=plot_df,  # Using filtered data here to match counts
    x='Bin',
    y='correlation',
    color=plot_color,
    order=order[:-1],
    ax=ax,
    linecolor='k',
    linewidth=2,
    saturation=1, 
    showcaps=False
)

plt.ylim(-0.5, 1)

# 4. Calculate counts for just this study per bin
counts_series = plot_df.groupby('Bin', observed=False)['correlation'].count()

# Unified single row height for total counts
y_top = 1.02    

for i, bin_name in enumerate(order[:-1]):
    if bin_name in counts_series.index:
        count_val = counts_series.loc[bin_name]
        
        if not pd.isna(count_val) and count_val > 0:
            ax.text(i, y_top, f'{int(count_val)}', 
                    ha='center', va='bottom', fontsize=14, 
                    fontweight='bold', color=sns.color_palette('tab20b')[-3])

# Remove legend as it's now a single-category plot
# plt.legend() removed

ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.1))
ax.grid(which='minor', axis='y', linestyle=':', alpha=0.5) 

plt.xlabel('River width (m)', fontsize=18, fontweight='bold')
plt.ylabel(r"Pearson's ${r}$", fontsize=18, fontweight='bold')
plt.xticks(rotation=-45, fontsize=16, fontweight='semibold')
plt.yticks(fontsize=16, fontweight='semibold')

plt.savefig(f'{save_dir}filtered_nhd_widths_corr.png', dpi=300, bbox_inches='tight')

In [ ]:
# 1. Filter data upfront for just the requested study
target_study = 'This study: filtered sites (NHDPlus HR widths)'
study_df = corr_combo[corr_combo['data'] == target_study]

study_df = study_df.drop(columns='data')
study_df.loc[study_df['Bin'].isin(order[4:]), 'Bin'] = '> 50'

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

# 2. Get unique bins and sort them safely
def sort_bins(x):
    if x == "> 50":
        return float("inf")  # Forces '> 50' to always sit at the end
    try:
        # Extracts the lower bound of the interval string, e.g., '(10, 20]' -> 10
        return float(x.replace("(", "").replace("]", "").split(",")[0])
    except ValueError:
        return x


bins = sorted(study_df["Bin"].unique(), key=sort_bins)

# 2. Group correlations by Bin
grouped_data = {bin_label: group["correlation"].values for bin_label, group in study_df.groupby("Bin")}

# 3. Initialize an empty DataFrame for the pairwise p-values
ks_posthoc_matrix = pd.DataFrame(np.nan, index=bins, columns=bins)

# 4. Perform pairwise KS tests between all bins
for i, bin1 in enumerate(bins):
    for j, bin2 in enumerate(bins):
        # We only need the lower triangle to match your plot's mask setup
        if i >= j: 
            data1 = grouped_data[bin1]
            data2 = grouped_data[bin2]
            
            # Run the KS test and extract the p-value
            _, p_val = ks_2samp(data1, data2)
            ks_posthoc_matrix.loc[bin1, bin2] = p_val

In [ ]:
# K-S PLOT

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# 1. Setup categorical data (1 for Sig, 0 for Not Sig)
sig_matrix = (ks_posthoc_matrix < 0.05).astype(int)

# Use k=0 to match the mask style of the second plot (hides diagonal and upper triangle)
mask = np.triu(np.ones_like(sig_matrix, dtype=bool), k=0)

# 2. Define colors and palette from the second plot
colors = ["lemonchiffon", "royalblue"]  # [Not Sig, Significant]
cmap = sns.color_palette(colors)

# Setup figure
plt.figure(figsize=(10, 8))

# 3. Plot with second plot's aesthetics (square cells, clean lines, bold text)
ax = sns.heatmap(
    sig_matrix,
    mask=mask,
    annot=ks_posthoc_matrix,  # Keep original p-values as text
    fmt=".3f",
    cmap=cmap,
    cbar=False,  # Turn off colorbar
    square=True,  # Keeps cells perfectly square
    annot_kws={
        "size": 10,
        "weight": "bold",
    },  # Bold annotations for contrast
)

# Optional labels to match the axis label style of the second plot
ax.set_xlabel("X Label Here", fontsize=14)
ax.set_ylabel("Y Label Here", fontsize=14)
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)

# 4. Create the formatted legend matching the second plot's font properties
not_sig_patch = mpatches.Patch(
    color=colors[0], label="Not Significant (p ≥ 0.05)"
)
sig_patch = mpatches.Patch(color=colors[1], label="Significant (p < 0.05)")

plt.legend(
    handles=[not_sig_patch, sig_patch],
    loc="upper center",
    bbox_to_anchor=(0.5, 1.1),  # Centers it cleanly above the single plot
    ncol=2,
    frameon=False,
    prop={"size": 12, "weight": "semibold"},
)

plt.xlabel('River width (m)')
plt.ylabel('River width (m)')

# plt.tight_layout()
# plt.show()

save_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results_figures/'
plt.savefig(f'{save_dir}ks_test_corr.png', dpi=300, bbox_inches='tight')

In [ ]:
# Correlation plot
fig, ax = plt.subplots(figsize=(10, 6))
sns.set_style('whitegrid')

# my_palette = [sns.color_palette("tab20b")[13],
#               'orange', #'#00FF7F',
#               'lightskyblue']

my_palette = [
    # '#8DAE91',
    '#668D6B',
    # '#7570B3',
    # 'orangered',
    'deeppink',
    sns.color_palette("tab20b")[13]]

plot_df = corr_combo.groupby(['Bin', 'data'], observed=False).filter(lambda x: len(x) >= 20)
plot_df = plot_df[plot_df['Bin'].isin(order[0:-3])]

# site_counts = df.groupby('Bin')['site_id'].nunique()

sns.boxplot(
    data=corr_combo,#[~corr_combo['site_id'].isin(to_exclude)],
    x='Bin',
    y='correlation',
    hue='data',
    palette=my_palette,
    order=order[2:-3],
    ax=ax,
    linecolor='k',
    # fliersize=0, 
    saturation=1, 
    showcaps=False)

plt.ylim(-0.6, 1)

# plot_df = corr_combo#[~corr_combo['site_id'].isin(to_exclude)]
counts_df = plot_df.groupby(['Bin', 'data'], observed=False)['correlation'].count().unstack()

y_top = 1.15    # Top number
y_middle = 1.1 # Bottom number (0.1 gap is safe for size 9 font)
y_bottom = 1.05

for i, bin_name in enumerate(order[2:-3]): # [0:10]
    if bin_name in counts_df.index:
        row = counts_df.loc[bin_name]
        
        # Study 1 (Top) - using the first color from your palette
        if not pd.isna(row.iloc[0]):
            ax.text(i, y_top, f'{int(row.iloc[1])}', 
                    ha='center', va='top', fontsize=9, 
                    fontweight='bold'
                    , color=my_palette[0]
                   )
            
        # Study 2 (middle) - using the second color
        if not pd.isna(row.iloc[1]):
            ax.text(i, y_middle, f'{int(row.iloc[0])}', 
                    ha='center', va='top', fontsize=9, 
                    fontweight='bold'
                    , color=my_palette[1]
                   )

        # Study 2 (bottom) - using the second color
        if not pd.isna(row.iloc[2]):
            ax.text(i, y_bottom, f'{int(row.iloc[0])}', 
                    ha='center', va='top', fontsize=9, 
                    fontweight='bold'
                    , color=my_palette[2]
                   )

plt.legend(
    # title='Study', 
           loc='lower right')

ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.1))
ax.grid(which='minor', axis='y', linestyle=':', alpha=0.5) 

# plt.title('"This study, matched gauges" mapped to SWORD widths', y=1.15)
plt.xlabel('River width (m)');
plt.ylabel(r'Median $r$');
plt.xticks(rotation=-45);

plt.savefig(f'{save_dir}relative_three_sword.png', dpi=300, bbox_inches='tight')

In [ ]:
# Bias plot
fig, ax = plt.subplots(figsize=(10, 6))
sns.set_style('whitegrid')

my_palette = [
    # '#8DAE91',
    '#668D6B',
    # '#7570B3',
    # 'orangered',
    'deeppink',
    sns.color_palette("tab20b")[13]]

plot_df = bias_combo.groupby(['Bin', 'data'], observed=False).filter(lambda x: len(x) > 20)
plot_df = plot_df[plot_df['Bin'].isin(order[2:-3])]

counts_df = plot_df.groupby(['Bin', 'data'], observed=False)['wse_diff_median'].count().unstack()

sns.boxplot(
    data=plot_df,#[~corr_combo['site_id'].isin(to_exclude)],
    x='Bin',
    y='wse_diff_median',
    hue='data',
    palette=my_palette,
    order=order[2:-3],
    ax=ax,
    linecolor='k',
    # fliersize=0, 
    legend=False,
    saturation=1, 
    showcaps=False)

y_top = 0.59    # Top number
y_middle = 0.56 # Bottom number (0.1 gap is safe for size 9 font)
y_bottom = 0.53

for i, bin_name in enumerate(order[2:-3]): # [0:10]
    if bin_name in counts_df.index:
        row = counts_df.loc[bin_name]
        
        # Study 1 (Top) - using the first color from your palette
        if not pd.isna(row.iloc[0]):
            ax.text(i, y_top, f'{int(row.iloc[1])}', 
                    ha='center', va='top', fontsize=9, 
                    fontweight='bold'
                    , color=my_palette[0]
                   )
            
        # Study 2 (middle) - using the second color
        if not pd.isna(row.iloc[1]):
            ax.text(i, y_middle, f'{int(row.iloc[0])}', 
                    ha='center', va='top', fontsize=9, 
                    fontweight='bold'
                    , color=my_palette[1]
                   )

        # Study 2 (bottom) - using the second color
        if not pd.isna(row.iloc[2]):
            ax.text(i, y_bottom, f'{int(row.iloc[0])}', 
                    ha='center', va='top', fontsize=9, 
                    fontweight='bold'
                    , color=my_palette[2]
                   )

ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.05))
ax.grid(which='minor', axis='y', linestyle=':', alpha=0.5) 

# plt.title('"This study, matched gauges" mapped to SWORD widths', y=1.15)
plt.xlabel('River width (m)');
plt.ylabel('Median absolute bias (m)');
plt.xticks(rotation=-45);

plt.ylim(-0.5,0.5)

plt.savefig(f'{save_dir}relative_bias_three_sword_zoom.png', dpi=300, bbox_inches='tight')